# 04 · Transformers (DistilBERT, RoBERTa) + class-imbalance experiments

**Part A** – fine-tune `distilbert-base-uncased` and `roberta-base` as baselines (Hugging Face `Trainer`, `max_len=64`, 3 epochs).
**Part B** – pick the best transformer **by validation macro-F1** and run the imbalance study on it (same model, same seed, same splits):

| Condition | What changes |
|---|---|
| `baseline` | nothing |
| `class_weight` | loss weight `w_c = N / (|E|·n_c)` (changes the objective, not the data) |
| `oversample` | random duplication of minority classes in **train only** |
| `augment` | synonym replacement + back-translation (EN→FR→EN, MarianMT) of minority classes in **train only** |
| `augment+class_weight` | combination |

**Part C** – promote the best configuration to `models/<model>/final/` and write `models/final_model.json` (used by notebook 06 and the app).

> 🖥️ Use a **GPU runtime**. Rough Colab-T4 cost: DistilBERT ≈ 10–15 min / run, RoBERTa ≈ 30 min / run. Checkpoints go to local disk, only final weights are copied to Drive.
> Start with `SEEDS=[42]`; extend to `[42, 43, 44]` if time allows.

In [ ]:
!pip -q install transformers datasets accelerate sentencepiece sacremoses nltk scikit-learn seaborn

In [ ]:
import os, sys, json, random, time, warnings, re, gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

# ---- Project root: Google Drive folder on Colab, repo root locally ----------
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/mental-health-nlp")   # <- change if you keep the repo elsewhere
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

for d in ["data/raw", "data/interim", "data/processed", "data/splits",
          "models/classical", "models/bilstm", "models/distilbert", "models/roberta",
          "results/figures", "results/reports"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

SEED = 42
EMOTIONS = ["Joy", "Sadness", "Anger", "Fear", "Anxiety", "Stress", "Surprise", "Neutral"]
LABEL2ID = {e: i for i, e in enumerate(EMOTIONS)}
ID2LABEL = {i: e for e, i in LABEL2ID.items()}
NEGATIVE = ["Sadness", "Anger", "Fear", "Anxiety", "Stress"]      # N in the paper (Sec. 5.6)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

set_seed()
print("ROOT =", ROOT)

In [ ]:
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score,
                             confusion_matrix, classification_report)

METRICS_CSV = ROOT / "results/model_metrics.csv"
LATENCY_CSV = ROOT / "results/latency_results.csv"
LABEL_IDS = list(range(len(EMOTIONS)))

def slug(condition):
    """File-system friendly condition name ('augment+class_weight' -> 'augment-class_weight')."""
    return condition.replace("+", "-")

def _upsert(path, row, keys):
    df = pd.read_csv(path) if path.exists() else pd.DataFrame()
    if len(df):
        mask = np.ones(len(df), dtype=bool)
        for k in keys:
            mask &= (df[k].astype(str) == str(row[k])).values
        df = df[~mask]
    pd.concat([df, pd.DataFrame([row])], ignore_index=True).to_csv(path, index=False)

def evaluate(y_true, y_pred, model, condition="baseline", seed=SEED, extra=None, log=True):
    """Compute all metrics, optionally upsert into results/model_metrics.csv."""
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=LABEL_IDS,
                                                 average="macro", zero_division=0)
    per = f1_score(y_true, y_pred, labels=LABEL_IDS, average=None, zero_division=0)
    row = dict(model=model, condition=condition, seed=seed,
               accuracy=accuracy_score(y_true, y_pred),
               precision_macro=p, recall_macro=r, macro_f1=f, min_class_f1=per.min())
    row.update({f"f1_{e}": v for e, v in zip(EMOTIONS, per)})
    if extra:
        row.update(extra)
    if log:
        _upsert(METRICS_CSV, row, ["model", "condition", "seed"])
    return row

def log_latency(model, device, ms_per_text, note=""):
    _upsert(LATENCY_CSV, dict(model=model, device=device, ms_per_text=round(float(ms_per_text), 2), note=note),
            ["model", "device", "note"])

def plot_cm(y_true, y_pred, title, path=None, normalize=True):
    cm = confusion_matrix(y_true, y_pred, labels=LABEL_IDS)
    if normalize:
        cm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(7.5, 6))
    sns.heatmap(cm, annot=True, fmt=".2f" if normalize else "d", cmap="Blues",
                xticklabels=EMOTIONS, yticklabels=EMOTIONS, ax=ax, cbar=False)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    plt.tight_layout()
    if path:
        fig.savefig(path, dpi=150)
    plt.show()

In [ ]:
SPLIT_DIR = ROOT / "data/splits"
train = pd.read_csv(SPLIT_DIR / "train.csv")
val   = pd.read_csv(SPLIT_DIR / "val.csv")
test  = pd.read_csv(SPLIT_DIR / "test.csv")
for d in (train, val, test):
    d["text_tf"] = d["text_tf"].fillna("").astype(str)
    d["text_clean"] = d["text_clean"].fillna("").astype(str)
print({k: len(v) for k, v in dict(train=train, val=val, test=test).items()})
train["label"].value_counts().reindex(EMOTIONS)

## 1 · Configuration, class weights, custom Trainer

In [ ]:
import torch, torch.nn as nn, shutil, nltk
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer,
                          DataCollatorWithPadding, EarlyStoppingCallback, set_seed as hf_set_seed)
nltk.download("wordnet", quiet=True); nltk.download("stopwords", quiet=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "!! no GPU - this will be very slow !!")

MODELS = {"distilbert": "distilbert-base-uncased", "roberta": "roberta-base"}
MAX_LEN, EPOCHS, BATCH, LR = 64, 3, 32, 2e-5
SEEDS = [42]
CKPT_DIR = Path("/content/ckpt") if IN_COLAB else ROOT / "models" / "_ckpt"      # big temp checkpoints stay off Drive
CKPT_DIR.mkdir(parents=True, exist_ok=True)

def class_weight_tensor(df):
    n_c = df["label_id"].value_counts().reindex(range(len(EMOTIONS))).fillna(1).values
    return torch.tensor(len(df) / (len(EMOTIONS) * n_c), dtype=torch.float)

class WeightedTrainer(Trainer):
    """Cross-entropy with optional per-class weights (w_c = N / (|E| n_c))."""
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        w = self.class_weights.to(outputs.logits.device) if self.class_weights is not None else None
        loss = nn.CrossEntropyLoss(weight=w)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    pred = logits.argmax(-1)
    return {"macro_f1": f1_score(labels, pred, average="macro"), "accuracy": accuracy_score(labels, pred)}

def to_ds(df, tok):
    ds = Dataset.from_dict({"text": df["text_tf"].tolist(), "labels": df["label_id"].astype(int).tolist()})
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN), batched=True, remove_columns=["text"])

## 2 · Imbalance mitigation helpers (training split only)
Target size per class = **median class count** of the training set. Minority classes are brought up to that size.
Augmented rows are cached in `data/interim/` so back-translation runs only once.

In [ ]:
from nltk.corpus import wordnet, stopwords
STOP = set(stopwords.words("english"))
TARGET = int(train["label_id"].value_counts().median())
MAX_AUG_PER_CLASS = 4000
print("per-class target size:", TARGET)

def random_oversample(df, seed):
    rng = np.random.default_rng(seed); parts = [df]
    for c, n in df["label_id"].value_counts().items():
        if n < TARGET:
            parts.append(df[df.label_id == c].sample(TARGET - n, replace=True, random_state=int(rng.integers(1e9))))
    return pd.concat(parts, ignore_index=True)

def synonym_replace(text, n=2, rng=None):
    rng = rng or random.Random(SEED)
    words = text.split()
    cand = [i for i, w in enumerate(words) if w.isalpha() and w.lower() not in STOP and len(w) > 3]
    rng.shuffle(cand); done = 0
    for i in cand:
        syns = {l.name().replace("_", " ") for s in wordnet.synsets(words[i].lower()) for l in s.lemmas()
                if l.name().lower() != words[i].lower() and "_" not in l.name()}
        if syns:
            words[i] = rng.choice(sorted(syns)); done += 1
        if done >= n: break
    return " ".join(words)

def _translate(texts, model_name, bs=64):
    from transformers import MarianMTModel, MarianTokenizer
    tk = MarianTokenizer.from_pretrained(model_name)
    mt = MarianMTModel.from_pretrained(model_name).to(DEVICE).eval()
    out = []
    for i in range(0, len(texts), bs):
        batch = tk(texts[i:i + bs], return_tensors="pt", padding=True, truncation=True, max_length=96).to(DEVICE)
        with torch.no_grad():
            gen = mt.generate(**batch, num_beams=2, max_new_tokens=96)
        out += tk.batch_decode(gen, skip_special_tokens=True)
    del mt; gc.collect(); torch.cuda.empty_cache()
    return out

def back_translate(texts):
    return _translate(_translate(texts, "Helsinki-NLP/opus-mt-en-fr"), "Helsinki-NLP/opus-mt-fr-en")

def make_augmented(df, seed):
    """Half of the synthetic rows via synonym replacement, half via back-translation. Cached on disk."""
    cache = ROOT / f"data/interim/augmented_train_s{seed}.csv"
    if cache.exists():
        print("loading cached augmentation:", cache.name); return pd.read_csv(cache)
    rng = random.Random(seed); new = []
    for c, n in df["label_id"].value_counts().items():
        need = min(max(TARGET - n, 0), MAX_AUG_PER_CLASS)
        if need == 0: continue
        src = df[df.label_id == c].sample(need, replace=True, random_state=seed)["text_tf"].tolist()
        half = need // 2
        syn = [synonym_replace(t, rng=rng) for t in src[:half]]
        bt = back_translate(src[half:])
        new += [dict(id=f"aug_{c}_{i}", source="augmented", text=t, text_tf=t, text_clean="", label=ID2LABEL[c], label_id=c)
                for i, t in enumerate(syn + bt)]
        print(f"  {ID2LABEL[c]:9s}: +{need} synthetic ({half} synonym, {need - half} back-translated)")
    aug = pd.DataFrame(new)
    aug = aug[~aug["text_tf"].isin(set(df["text_tf"]))].drop_duplicates("text_tf")    # drop unchanged copies
    aug.to_csv(cache, index=False)
    return aug

def build_train_df(condition, seed):
    parts = condition.split("+")
    if "oversample" in parts: return random_oversample(train, seed)
    if "augment" in parts:    return pd.concat([train, make_augmented(train, seed)], ignore_index=True)
    return train

In [ ]:
# Manual spot-check of augmented samples (paper Sec. 5.4) — run once after the first `augment` condition has generated the cache,
# or call it now (takes a few minutes of back-translation on GPU).
def spot_check(n=3, seed=SEED):
    aug = make_augmented(train, seed)
    for c in ["Anxiety", "Fear", "Surprise", "Stress"]:
        print(f"\n--- {c} ---")
        for t in aug[aug.label == c].sample(min(n, (aug.label == c).sum()), random_state=1)["text_tf"]:
            print("  *", t[:180])
# spot_check()

## 3 · One experiment = (model, condition, seed)

In [ ]:
def run_experiment(key, condition, seed=SEED, lr=LR, epochs=EPOCHS, bs=BATCH, save=True):
    t0 = time.time()
    hf_set_seed(seed); set_seed(seed)
    tok = AutoTokenizer.from_pretrained(MODELS[key])
    tr_df = build_train_df(condition, seed)
    ds_tr, ds_va, ds_te = to_ds(tr_df, tok), to_ds(val, tok), to_ds(test, tok)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODELS[key], num_labels=len(EMOTIONS), id2label=ID2LABEL, label2id=LABEL2ID)

    args = TrainingArguments(
        output_dir=str(CKPT_DIR / f"{key}_{slug(condition)}_s{seed}"),
        num_train_epochs=epochs, per_device_train_batch_size=bs, per_device_eval_batch_size=128,
        learning_rate=lr, weight_decay=0.01, warmup_ratio=0.1, lr_scheduler_type="linear",
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
        fp16=torch.cuda.is_available(), report_to="none", seed=seed, logging_steps=200)

    use_w = "class_weight" in condition.split("+")
    trainer = WeightedTrainer(model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
                              data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics,
                              class_weights=class_weight_tensor(tr_df) if use_w else None,
                              callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    trainer.train()

    val_f1 = trainer.evaluate(ds_va)["eval_macro_f1"]            # best epoch (load_best_model_at_end)
    logits = trainer.predict(ds_te).predictions
    probs = torch.softmax(torch.tensor(logits), -1).numpy(); pred = probs.argmax(1)
    row = evaluate(test["label_id"].values, pred, model=key, condition=condition, seed=seed,
                   extra=dict(val_macro_f1=val_f1, train_size=len(tr_df), minutes=round((time.time() - t0) / 60, 1)))
    pd.DataFrame({"id": test["id"], "source": test["source"], "text": test["text"], "true": test["label"],
                  "pred": [ID2LABEL[i] for i in pred], "conf": probs.max(1)}
                 ).to_csv(ROOT / f"results/predictions_{key}_{slug(condition)}.csv", index=False)
    if save:
        out = ROOT / "models" / key / f"{slug(condition)}_s{seed}"
        trainer.save_model(str(out)); tok.save_pretrained(str(out))
    print(f"[{key} | {condition} | s{seed}] val F1={val_f1:.4f}  test acc={row['accuracy']:.4f} "
          f"macro-F1={row['macro_f1']:.4f} min-class-F1={row['min_class_f1']:.4f}  ({(time.time()-t0)/60:.1f} min)")
    del trainer, model; gc.collect(); torch.cuda.empty_cache()
    return row

## Part A · Baselines — DistilBERT and RoBERTa

In [ ]:
for key in ["distilbert", "roberta"]:
    for s in SEEDS:
        run_experiment(key, "baseline", seed=s)

In [ ]:
m = pd.read_csv(METRICS_CSV)
tb = m[m.model.isin(MODELS) & (m.condition == "baseline")]
display(tb[["model", "seed", "val_macro_f1", "accuracy", "macro_f1", "min_class_f1"]].round(4))
BEST_KEY = tb.groupby("model")["val_macro_f1"].mean().idxmax()          # choose on VALIDATION, not test
print("Best transformer (by validation macro-F1):", BEST_KEY)

## Part B · Class-imbalance study on the best transformer
Same model, same seed(s), same untouched validation/test sets. Only the *training* data / loss change.

In [ ]:
CONDITIONS = ["class_weight", "oversample", "augment", "augment+class_weight"]
for cond in CONDITIONS:
    for s in SEEDS:
        run_experiment(BEST_KEY, cond, seed=s)

In [ ]:
m = pd.read_csv(METRICS_CSV); bm = m[m.model == BEST_KEY]
f1cols = [f"f1_{e}" for e in EMOTIONS]
imb = bm.groupby("condition")[f1cols + ["macro_f1", "accuracy", "min_class_f1"]].mean()
imb = imb.loc[["baseline"] + [c for c in CONDITIONS if c in imb.index]].rename(columns=lambda c: c.replace("f1_", "F1 "))
imb.round(4).to_csv(ROOT / "results/imbalance_results.csv")
display(imb.round(3))

delta = imb.sub(imb.loc["baseline"], axis=1).drop(index="baseline")
print("\nChange vs. baseline (positive = improvement):")
display(delta.round(3).style.background_gradient(cmap="RdYlGn", axis=None, vmin=-0.15, vmax=0.15))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.8))
plot_df = imb[[f"F1 {e}" for e in EMOTIONS]].T; plot_df.index = EMOTIONS
plot_df.plot(kind="bar", ax=ax, width=0.82, rot=0)
ax.set_ylabel("Test F1"); ax.set_ylim(0, 1); ax.legend(title="condition", ncol=3, fontsize=8)
ax.set_title(f"Per-class F1 by imbalance-mitigation condition — {BEST_KEY}")
plt.tight_layout(); plt.savefig(ROOT / "results/figures/04_imbalance_per_class_f1.png", dpi=150); plt.show()

## Part C · Promote the best configuration
Selection uses **validation** macro-F1 (never the test set). The chosen weights are copied to `models/<model>/final/` and described in `models/final_model.json`.

In [ ]:
cand = bm.copy()
best_row = cand.sort_values("val_macro_f1", ascending=False).iloc[0]
best_cond, best_seed = best_row["condition"], int(best_row["seed"])
print(f"Selected: {BEST_KEY} | condition={best_cond} | seed={best_seed} | val macro-F1={best_row['val_macro_f1']:.4f} "
      f"| test macro-F1={best_row['macro_f1']:.4f}")

src = ROOT / "models" / BEST_KEY / f"{slug(best_cond)}_s{best_seed}"
dst = ROOT / "models" / BEST_KEY / "final"
shutil.copytree(src, dst, dirs_exist_ok=True)
json.dump({"model": BEST_KEY, "hf_name": MODELS[BEST_KEY], "condition": best_cond, "seed": best_seed,
           "path": f"models/{BEST_KEY}/final", "max_len": MAX_LEN, "labels": EMOTIONS,
           "val_macro_f1": float(best_row["val_macro_f1"]), "test_macro_f1": float(best_row["macro_f1"])},
          open(ROOT / "models/final_model.json", "w"), indent=2)
print("final model ->", dst)

## Cross-domain view (RQ3) and latency

In [ ]:
p = pd.read_csv(ROOT / f"results/predictions_{BEST_KEY}_{slug(best_cond)}.csv")
rows = []
for src_name, g in p.groupby("source"):
    rows.append(dict(source=src_name, n=len(g), accuracy=(g.true == g.pred).mean(),
                     macro_f1=f1_score(g.true, g.pred, labels=[e for e in EMOTIONS if (g.true == e).any()],
                                       average="macro", zero_division=0)))
display(pd.DataFrame(rows).round(4))
st = p[p.true == "Stress"]
print("Stress recall:", round((st.pred == "Stress").mean(), 3), "| most common wrong predictions:",
      st[st.pred != "Stress"].pred.value_counts().head(3).to_dict())

In [ ]:
tok = AutoTokenizer.from_pretrained(str(dst))
mdl = AutoModelForSequenceClassification.from_pretrained(str(dst)).eval()
enc = tok(test["text_tf"].iloc[0], truncation=True, max_length=MAX_LEN, return_tensors="pt")
for dev in ["cpu"] + (["cuda"] if torch.cuda.is_available() else []):
    mm, e = mdl.to(dev), {k: v.to(dev) for k, v in enc.items()}
    with torch.no_grad():
        for _ in range(10): mm(**e)
        if dev == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(100): mm(**e)
        if dev == "cuda": torch.cuda.synchronize()
    ms = 1000 * (time.perf_counter() - t0) / 100
    log_latency(BEST_KEY, "gpu" if dev == "cuda" else "cpu", ms, note="inference only (tokenised input), batch=1")
    print(f"{BEST_KEY} on {dev}: {ms:.1f} ms/text")

### ✅ Done
Artifacts: `models/<model>/<condition>_s<seed>/`, `models/<model>/final/`, `models/final_model.json`, `results/imbalance_results.csv`, `results/predictions_*.csv`.

*Optional extension (RQ3):* continue fine-tuning the final model on Dreaddit-only data (domain adaptation) and compare Stress-class F1.
If the second transformer should also get the imbalance study, set `BEST_KEY` manually and re-run Part B.

Next: **`05_error_analysis.ipynb`**.